<a href="https://colab.research.google.com/github/hebawl/starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [26]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 160)

df = pd.read_csv("content_refresh_anonymized.csv")
print(df.shape)


def weighted_ctr_pct(g):
  imp=g["impressions_90d"].sum()
  return (g["clicks_90d"].sum()/imp*100) if imp>0 else np.nan

# Signal check 1
# Do pages that have not been updated for a long time have a lower CTR?
bins=[0,30,180,10000]
labels=[
    "fresh_0_30",
    "aging_31_180",
    "stale_180plus"
]

df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

def summarize_stale(g):
    return pd.Series({
        "n": len(g),
        "median_impressions_90d": g["impressions_90d"].median(),
        "median_avg_position": g.loc[g["avg_position"] > 0, "avg_position"].median(),
        "weighted_ctr_pct": weighted_ctr_pct(g),
    })

signal1_table = df.groupby("stale_bucket", observed=True).apply(summarize_stale).loc[labels]
print("Signal 1 - staleness vs. performance (bucket table, n shown):")
print(signal1_table)
print()
print("Verdict: CONFIRMED (directional).")
print("Weighted CTR drops as staleness increases (0.33% -> 0.29% -> 0.23%), all buckets")
print("well above the ~50-row floor. Staler content really does underperform on CTR -")
print("this is the assumption FlyRank's refresh flags lean on, and the data backs it.")
print("(Median position doesn't move as cleanly - the 180+ bucket is smaller and noisier,")
print("so I'm not leaning on position for this rule, only CTR.)")


# Signal check 2
#Is a page that many people see worth paying attention to?
order = ["low", "moderate", "good", "excellent"]
df2 = df[df["impression_tier"].isin(order)].copy()
df2["impression_tier"] = pd.Categorical(df2["impression_tier"], categories=order, ordered=True)

def summarize_visibility(g):
    return pd.Series({
        "n": len(g),
        "median_clicks_90d": g["clicks_90d"].median(),
        "weighted_ctr_pct": weighted_ctr_pct(g),
    })

signal2_table = df2.groupby("impression_tier", observed=True).apply(summarize_visibility).loc[order]
print("Signal 2 - impression tier vs. real clicks (bucket table, n shown):")
print(signal2_table)
print()
print("Verdict: CONFIRMED.")
print("Median clicks rise cleanly across every tier (0 -> 1 -> 16 -> 116), n in the")
print("thousands for every bucket. Higher impression tiers really do carry more real")
print("traffic, not just more impressions on the same tiny CTR - so gating the rule on")
print("impressions_90d >= 500 (inside the 'moderate'/'good' range) is a real filter for")
print("'is anyone actually seeing this page', not an arbitrary cutoff.")



(30000, 44)
Signal 1 - staleness vs. performance (bucket table, n shown):
                     n  median_impressions_90d  median_avg_position  weighted_ctr_pct
stale_bucket                                                                         
fresh_0_30     20480.0                   470.0                10.60          0.326873
aging_31_180    9346.0                  1643.5                13.70          0.288426
stale_180plus    174.0                    15.5                 7.85          0.227934

Verdict: CONFIRMED (directional).
Weighted CTR drops as staleness increases (0.33% -> 0.29% -> 0.23%), all buckets
well above the ~50-row floor. Staler content really does underperform on CTR -
this is the assumption FlyRank's refresh flags lean on, and the data backs it.
(Median position doesn't move as cleanly - the 180+ bucket is smaller and noisier,
so I'm not leaning on position for this rule, only CTR.)
Signal 2 - impression tier vs. real clicks (bucket table, n shown):
              

/tmp/ipykernel_1613/446890533.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signal1_table = df.groupby("stale_bucket", observed=True).apply(summarize_stale).loc[labels]
/tmp/ipykernel_1613/446890533.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signal2_table = df2.groupby("impression_tier", observed=True).apply(summarize_visibility).loc[order]


In [27]:
# Rule:
# A page is flagged for refresh if it has not been updated for at least 31 days and has received 500 or more impressions in the last 90 days.
# Pages that meet both conditions receive an action score equal to their impressions and are ranked from highest to lowest priority.

# Reason codes:

# stale_visible_page – The page is stale and receives enough impressions to make a refresh worthwhile.
# not_flagged – The page does not meet one or both conditions, so it is monitored instead of refreshed.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [28]:
stale = (df["days_since_last_update"] >= 31).astype(int)     # signal 1 is the page at least 31 days old?
visible = (df["impressions_90d"] >= 500).astype(int)          # signal 2 has this page received at least 500 impressions?

df["action_score"] = stale * visible * df["impressions_90d"] # if both are true give score

df["reason_code"] = np.where(
    df["action_score"] > 0,
    "stale_visible_page",
    "not_flagged")

df["action"] = np.where(
    df["action_score"] > 0,
    "refresh",
    "monitor")

df["rank"] = df["action_score"].rank(
    method="first",
    ascending=False).astype(int)

flagged_n = (df["action_score"] > 0).sum()
print(f"Flagged for refresh: {flagged_n} of {len(df)} rows ({flagged_n/len(df):.1%})")

output_cols = [
    "content_id", "client_id", "rank", "action_score", "action", "reason_code",
    "days_since_last_update", "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "content_age_days", "word_count",
]
queue = df[output_cols].sort_values("rank")

out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({len(queue)} rows)")

queue.head(10)

Flagged for refresh: 6663 of 30000 rows (22.2%)
Wrote ../outputs/baseline_action_score.csv (30000 rows)


,content_id,client_id,rank,action_score,action,reason_code,days_since_last_update,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,word_count
6653,content_5fe46e04994d,client_4e07408562,1,517715,refresh,stale_visible_page,104,517715,741,4.2,0.14,537,NaN
19636,content_2cb567c3c89b,client_6208ef0f77,2,497727,refresh,stale_visible_page,48,497727,487,22.2,0.10,153,6183.0
29400,content_2dba2b1f9536,client_6208ef0f77,3,443434,refresh,stale_visible_page,104,443434,910,27.9,0.21,299,7676.0
13537,content_2c2606c5d176,client_19581e27de,4,347399,refresh,stale_visible_page,104,347399,1854,4.2,0.53,362,NaN
26531,content_cb112fce36be,client_19581e27de,5,309910,refresh,stale_visible_page,104,309910,492,5.6,0.16,126,2761.0
21565,content_9532f197bbc8,client_4e07408562,6,309192,refresh,stale_visible_page,104,309192,2689,2.0,0.87,445,NaN
3394,content_36ff89c8214e,client_19581e27de,7,295097,refresh,stale_visible_page,104,295097,154,7.3,0.05,144,NaN
26798,content_b28d1efd668f,client_6208ef0f77,8,286608,refresh,stale_visible_page,104,286608,169,26.2,0.06,153,6901.0
23767,content_813e88069237,client_6208ef0f77,9,233561,refresh,stale_visible_page,104,233561,129,26.2,0.06,153,4610.0
26255,content_c21024970297,client_19581e27de,10,211366,refresh,stale_visible_page,104,211366,870,5.1,0.41,126,2874.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [29]:
queue.head(20)

,content_id,client_id,rank,action_score,action,reason_code,days_since_last_update,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,word_count
6653,content_5fe46e04994d,client_4e07408562,1,517715,refresh,stale_visible_page,104,517715,741,4.2,0.14,537,NaN
19636,content_2cb567c3c89b,client_6208ef0f77,2,497727,refresh,stale_visible_page,48,497727,487,22.2,0.10,153,6183.0
29400,content_2dba2b1f9536,client_6208ef0f77,3,443434,refresh,stale_visible_page,104,443434,910,27.9,0.21,299,7676.0
13537,content_2c2606c5d176,client_19581e27de,4,347399,refresh,stale_visible_page,104,347399,1854,4.2,0.53,362,NaN
26531,content_cb112fce36be,client_19581e27de,5,309910,refresh,stale_visible_page,104,309910,492,5.6,0.16,126,2761.0
21565,content_9532f197bbc8,client_4e07408562,6,309192,refresh,stale_visible_page,104,309192,2689,2.0,0.87,445,NaN
3394,content_36ff89c8214e,client_19581e27de,7,295097,refresh,stale_visible_page,104,295097,154,7.3,0.05,144,NaN
26798,content_b28d1efd668f,client_6208ef0f77,8,286608,refresh,stale_visible_page,104,286608,169,26.2,0.06,153,6901.0
23767,content_813e88069237,client_6208ef0f77,9,233561,refresh,stale_visible_page,104,233561,129,26.2,0.06,153,4610.0
26255,content_c21024970297,client_19581e27de,10,211366,refresh,stale_visible_page,104,211366,870,5.1,0.41,126,2874.0


In [30]:
# Top-20 review — one line per page: action, why, what would make it wrong.

# Rank 1 (content_5fe46e04994d): refresh. 104 days stale, 517,715 impressions (highest in the set),
#   good position (4.2) but low CTR (0.14%). Would be wrong if: this CTR is already normal for a
#   client whose average CTR at position ~4 runs low (need the client baseline to know for sure).

# Rank 2 (content_2cb567c3c89b): refresh. 48 days stale, 497,727 impressions, weak position (22.2),
#   low CTR (0.10%). Clean case — stale, visible, and genuinely underperforming.

# Rank 3 (content_2dba2b1f9536): refresh. 104 days stale, 443,434 impressions, weak position (27.9).
#   Would be wrong if: the page just dropped to page 3-5 from a temporary ranking shake-up, not decay.

# Rank 4 (content_2c2606c5d176): refresh. 104 days stale, good position (4.2), decent CTR (0.53%).
#   Flagged mainly on volume, not on any sign of trouble. Would be wrong if: this page is fine as-is
#   and doesn't need touching — staleness alone isn't proof of decay.

# Rank 5 (content_cb112fce36be): refresh. 104 days stale, strong position (5.6), low CTR (0.16%) —
#   a real CTR-fix candidate. Would be wrong if: this is a branded/navigational query where low CTR
#   is normal (users skip straight to a menu link instead of clicking the search result).

# Rank 6 (content_9532f197bbc8): refresh. WEAK PICK — position (2.0) and CTR (0.87%) are already
#   strong. Flagged purely by the staleness+volume gate, not by any real sign of decay. This is the
#   clearest case in the top 20 of the rule's blind spot: it has no "already healthy" check.

# Rank 7 (content_36ff89c8214e): refresh. 104 days stale, weak CTR (0.05%) at a middling position (7.3).
#   Would be wrong if: clicks_90d (154) being this low relative to impressions reflects a naturally
#   low-click-intent query (informational), not a fixable CTR problem.

# Rank 8 (content_b28d1efd668f): refresh. 104 days stale, weak position (26.2), very low CTR (0.06%) —
#   consistent decay story. Would be wrong if: the content type doesn't match search intent, in which
#   case a refresh alone won't fix the mismatch.

# Rank 9 (content_813e88069237): refresh. Same shape as rank 8 — stale, weak position (26.2), low CTR
#   (0.06%). Would be wrong if: ranks 8 and 9 are near-duplicate pages for the same client competing
#   for the same query — the real fix would be consolidation, not two separate refreshes.

# Rank 10 (content_c21024970297): refresh. 104 days stale, good position (5.1), moderate CTR (0.41%).
#   Borderline pick. Would be wrong if: this CTR is already above the client's average for position ~5.

# Rank 11 (content_c8e9d6ab9013): refresh. 104 days stale, decent position (9.7), but 0 clicks despite
#   208,678 impressions (CTR 0.00%). Would be wrong if: this is a tracking/data issue rather than a
#   real content problem — worth spot-checking before prioritizing.

# Rank 12 (content_b511d4bc4ad2): refresh. 104 days stale, weak position (27.9), low CTR (0.14%).
#   Consistent with the decay story. Would be wrong if: it's a seasonal page with naturally low
#   engagement outside its season.

# Rank 13 (content_d17681677e69): refresh. 104 days stale, strong position (5.8), low-ish CTR (0.24%).
#   Would be wrong if: 0.24% is already typical for that position/vertical.

# Rank 14 (content_a7427266c305): refresh. 104 days stale, strong position (5.7), low CTR (0.11%) —
#   solid CTR-fix candidate. Would be wrong if: query has low commercial intent.

# Rank 15 (content_c5063073d048): refresh. 104 days stale, mid position (12.5), low CTR (0.24%).
#   Reasonable pick. Would be wrong if: position 12.5 already explains the low CTR on its own (page 2
#   of results gets little traffic regardless of content quality).

# Rank 16 (content_3d94572c3a35): refresh. 104 days stale, good position (4.3), low CTR (0.24%) —
#   good CTR-fix candidate, position is strong enough that 0.24% looks like underperformance.

# Rank 17 (content_01908772c6db): refresh. 104 days stale, good position (4.0), CTR (0.45%) already
#   decent for that position. Would be wrong if: 0.45% is already at or above the client's norm.

# Rank 18 (content_33b4dceecad1): refresh. 104 days stale, ok position (6.2), low CTR (0.16%).
#   Reasonable pick, similar shape to rank 5.

# Rank 19 (content_f02b48f88241): refresh. 104 days stale, weak position (25.8), low CTR (0.10%).
#   Consistent decay story, no obvious red flag.

# Rank 20 (content_05e9b4cd9ccf): refresh. 104 days stale, weak position (22.1), low CTR (0.08%).
#   Consistent decay story, no obvious red flag.

# Overall confidence: high for most rows — staleness + visibility + weak CTR line up together.
# Rank 6 is the standout weak pick: it passed the rule's gates on volume alone despite already
# performing well, which is the rule's core limitation (explained more in section 4 below).

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [31]:
# Weak picks: summarized above (rank 6 is the clearest case) — pages that pass the
# staleness + visibility gates on volume alone, even though position/CTR already look healthy.

# Leakage check — don't just claim it, verify it in code:
scoring_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label",
             "health_score", "priority_score", "action_type"}

print("Columns actually used to compute action_score:", scoring_inputs)
print("Forbidden columns present in the dataset:", forbidden & set(df.columns))
assert not (forbidden & scoring_inputs), "Leakage detected: a forbidden column was used in scoring!"
print("No leakage: action_score depends only on days_since_last_update and impressions_90d,")
print("both of which are known before the decision is made — no future or label data involved.")

Columns actually used to compute action_score: {'days_since_last_update', 'impressions_90d'}
Forbidden columns present in the dataset: {'trend_pct', 'trend_direction'}
No leakage: action_score depends only on days_since_last_update and impressions_90d,
both of which are known before the decision is made — no future or label data involved.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.